# Exploratory Data Analysis

Notebook ini memeriksa bentuk data, distribusi target `damage_grade`, nilai yang hilang, dan ringkasan fitur sebelum pemodelan. Target memiliki lima kelas dan evaluasi kompetisi memakai Macro F1. Pastikan berkas kompetisi tersedia di `data/`.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {train.shape}; test: {test.shape}; sample: {sample.shape}')
print('Target:', train['damage_grade'].value_counts().sort_index().to_dict())
train.head()


## Kesesuaian kolom dan nilai hilang

Periksa apakah kolom fitur konsisten pada train dan test. `building_id` adalah identifier untuk submission, bukan fitur target.


In [ ]:
train_features = train.drop(columns=['damage_grade'])
print('Kolom hanya di train:', sorted(set(train_features.columns) - set(test.columns)))
print('Kolom hanya di test:', sorted(set(test.columns) - set(train_features.columns)))
missing = train.isna().sum().sort_values(ascending=False)
missing[missing.gt(0)].head(20)


## Distribusi target dan ringkasan fitur


In [ ]:
display(train['damage_grade'].value_counts().sort_index().rename('jumlah').to_frame())
display(train['damage_grade'].value_counts(normalize=True).sort_index().rename('proporsi').to_frame())
numeric_columns = train.select_dtypes(include='number').columns
train[numeric_columns].describe().T


## Kategori dan relasi sederhana dengan target

Tabel berikut menampilkan kategori dominan untuk beberapa fitur kategorikal. Ini bersifat deskriptif; keputusan preprocessing dan model tetap dievaluasi pada pembagian validasi.


In [ ]:
categorical_columns = train.select_dtypes(include=['object', 'category']).columns.tolist()
for column in categorical_columns[:8]:
    print(f'\n{column}')
    display(pd.crosstab(train[column], train['damage_grade'], normalize='index').head(12))
